# 第 31 章：卷积神经网络与图像局部特征

这个 notebook 用一个极小的星系 cutout 教学数据，把进入 CNN 之前最关键的直觉讲清楚：

- 为什么 raw pixels 直接摊平会对位置变化敏感
- 为什么卷积核更擅长抓局部模式
- 为什么 ReLU 和池化能让表示更稳健
- 怎样把这个教学版卷积 workflow 接到后续真正的 CNN 与迁移学习
- 为什么冻结局部特征提取器再接一个小型 target head，是迁移学习里很常见的第一步

教学说明：这里继续保持仓库当前风格，默认不依赖 NumPy、PyTorch 或图像库；环境里如果装了 `torch`，最后一格会给出一个可选的极小 CNN 对照。


In [ ]:
from __future__ import annotations

import csv
from collections import Counter
from pathlib import Path

DATA_PATH = Path("../../data/small/cnn_cutout_demo.csv").resolve()
IMAGE_SIZE = 6

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = []
    for raw in csv.DictReader(handle):
        parsed = {
            "sample_id": raw["sample_id"],
            "morphology_label": raw["morphology_label"],
            "split": raw["split"],
        }
        for row_index in range(IMAGE_SIZE):
            for col_index in range(IMAGE_SIZE):
                parsed[f"p{row_index}{col_index}"] = float(raw[f"p{row_index}{col_index}"])
        rows.append(parsed)

train_rows = [row for row in rows if row["split"] == "train"]
test_rows = [row for row in rows if row["split"] == "test"]


def image_from_row(row):
    return [[row[f"p{row_index}{col_index}"] for col_index in range(IMAGE_SIZE)] for row_index in range(IMAGE_SIZE)]


def flatten(matrix):
    return [value for line in matrix for value in line]


def ascii_heatmap(image):
    shades = " .:-=+*#%@"
    maximum = max(flatten(image)) or 1.0
    lines = []
    for line in image:
        chars = []
        for value in line:
            index = int(round((len(shades) - 1) * value / maximum))
            chars.append(shades[index])
        lines.append("".join(chars))
    return "\n".join(lines)


print(f"Loaded {len(rows)} cutout samples from {DATA_PATH.name}")
print("class counts:", dict(Counter(row["morphology_label"] for row in rows)))
print("train/test sizes:", len(train_rows), len(test_rows))
print("test ids:", [row["sample_id"] for row in test_rows])
print("Example cutouts:")
for sample_id in ["C01", "E01", "D01"]:
    row = next(item for item in rows if item["sample_id"] == sample_id)
    print(f"\n{sample_id} {row['morphology_label']}")
    print(ascii_heatmap(image_from_row(row)))


In [ ]:
import math


def centroid(vectors):
    return [sum(values) / len(values) for values in zip(*vectors)]


def euclidean_distance(left, right):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(left, right)))


def nearest_centroid(train_items, feature_fn, test_items):
    labels = sorted({item["morphology_label"] for item in train_items})
    centers = {}
    for label in labels:
        vectors = [feature_fn(item) for item in train_items if item["morphology_label"] == label]
        centers[label] = centroid(vectors)

    predictions = []
    for item in test_items:
        vector = feature_fn(item)
        ranked = sorted(
            ((label, euclidean_distance(vector, centers[label])) for label in labels),
            key=lambda pair: pair[1],
        )
        predictions.append({
            "sample_id": item["sample_id"],
            "true_label": item["morphology_label"],
            "predicted_label": ranked[0][0],
            "distances": {label: round(distance, 3) for label, distance in ranked},
        })
    return centers, predictions


def accuracy(predictions):
    return sum(item["true_label"] == item["predicted_label"] for item in predictions) / len(predictions)


def confusion(predictions):
    matrix = {}
    for item in predictions:
        truth = item["true_label"]
        predicted = item["predicted_label"]
        matrix.setdefault(truth, {})
        matrix[truth][predicted] = matrix[truth].get(predicted, 0) + 1
    return matrix


raw_centers, raw_predictions = nearest_centroid(
    train_rows,
    lambda item: flatten(image_from_row(item)),
    test_rows,
)

print("Raw-pixel nearest-centroid baseline:")
for item in raw_predictions:
    print(item["sample_id"], item["true_label"], "->", item["predicted_label"], item["distances"])
print("raw accuracy:", round(accuracy(raw_predictions), 3))
print("raw confusion matrix:", confusion(raw_predictions))


In [ ]:
kernels = {
    "core": [
        [-1.0, -1.0, -1.0],
        [-1.0, 6.0, -1.0],
        [-1.0, -1.0, -1.0],
    ],
    "horizontal": [
        [-1.0, -1.0, -1.0],
        [1.5, 1.5, 1.5],
        [-1.0, -1.0, -1.0],
    ],
    "double_lobe": [
        [1.0, -1.0, 1.0],
        [1.0, -3.0, 1.0],
        [1.0, -1.0, 1.0],
    ],
}


def conv2d_valid(image, kernel):
    output = []
    for row_index in range(len(image) - 2):
        row = []
        for col_index in range(len(image[0]) - 2):
            score = 0.0
            for kernel_row in range(3):
                for kernel_col in range(3):
                    score += image[row_index + kernel_row][col_index + kernel_col] * kernel[kernel_row][kernel_col]
            row.append(score)
        output.append(row)
    return output


def relu(matrix):
    return [[max(0.0, value) for value in line] for line in matrix]


def max_pool_2x2(feature_map):
    pooled = []
    for row_index in range(0, len(feature_map) - 1, 2):
        row = []
        for col_index in range(0, len(feature_map[0]) - 1, 2):
            row.append(max(
                feature_map[row_index][col_index],
                feature_map[row_index][col_index + 1],
                feature_map[row_index + 1][col_index],
                feature_map[row_index + 1][col_index + 1],
            ))
        pooled.append(row)
    return pooled


def summarize_kernel_response(image):
    summary = []
    detailed = {}
    for kernel_name, kernel in kernels.items():
        pooled = max_pool_2x2(relu(conv2d_valid(image, kernel)))
        pooled_values = flatten(pooled)
        summary.extend([max(pooled_values), sum(pooled_values) / len(pooled_values)])
        detailed[kernel_name] = pooled
    return summary, detailed


print("Pooled responses for representative training samples:")
for sample_id in ["C01", "E01", "D01"]:
    row = next(item for item in rows if item["sample_id"] == sample_id)
    feature_summary, detailed = summarize_kernel_response(image_from_row(row))
    print(f"\n{sample_id} {row['morphology_label']}")
    print("feature summary:", [round(value, 2) for value in feature_summary])
    for kernel_name, pooled in detailed.items():
        print(kernel_name, [[round(value, 2) for value in line] for line in pooled])


In [ ]:
def cnn_features(item):
    summary, _ = summarize_kernel_response(image_from_row(item))
    return summary


conv_centers, conv_predictions = nearest_centroid(train_rows, cnn_features, test_rows)

print("Convolution-feature nearest-centroid model:")
for item in conv_predictions:
    print(item["sample_id"], item["true_label"], "->", item["predicted_label"], item["distances"])
print("conv accuracy:", round(accuracy(conv_predictions), 3))
print("conv confusion matrix:", confusion(conv_predictions))
print("accuracy improvement over raw baseline:", round(accuracy(conv_predictions) - accuracy(raw_predictions), 3))


In [ ]:
print("Translation check on the double-source class:")
for sample_id in ["D01", "D03", "D04"]:
    row = next(item for item in rows if item["sample_id"] == sample_id)
    summary, detailed = summarize_kernel_response(image_from_row(row))
    print(sample_id, row["split"], [round(value, 2) for value in summary])
    print("double_lobe pooled =", [[round(value, 2) for value in line] for line in detailed["double_lobe"]])

print("Interpretation:")
print("  The raw-pixel baseline memorizes exact coordinates, so shifted double-source samples drift toward the edge-on centroid.")
print("  Convolution keeps looking for the same local motif at different positions, and max pooling keeps the strongest response.")
print("  That is the first practical reason CNN-style features outperform flattened pixels on spatial data.")


In [ ]:
TRANSFER_DATA_PATH = Path("../../data/small/cnn_transfer_learning_demo.csv").resolve()

with TRANSFER_DATA_PATH.open(newline="", encoding="utf-8") as handle:
    transfer_rows = []
    for raw in csv.DictReader(handle):
        parsed = {
            "sample_id": raw["sample_id"],
            "morphology_label": raw["morphology_label"],
            "domain_label": raw["domain_label"],
            "split_role": raw["split_role"],
        }
        for row_index in range(IMAGE_SIZE):
            for col_index in range(IMAGE_SIZE):
                parsed[f"p{row_index}{col_index}"] = float(raw[f"p{row_index}{col_index}"])
        transfer_rows.append(parsed)

source_train_rows = [row for row in transfer_rows if row["split_role"] == "source_train"]
target_adapt_rows = [row for row in transfer_rows if row["split_role"] == "target_adapt"]
target_test_rows = [row for row in transfer_rows if row["split_role"] == "target_test"]

print(f"Loaded {len(transfer_rows)} transfer-learning samples from {TRANSFER_DATA_PATH.name}")
print("split-role counts:", dict(Counter(row["split_role"] for row in transfer_rows)))
print("target-test ids:", [row["sample_id"] for row in target_test_rows])
print("Example source vs target cutouts:")
for sample_id in ["S_C1", "T_CA", "T_E1"]:
    row = next(item for item in transfer_rows if item["sample_id"] == sample_id)
    print(f"\n{sample_id} {row['domain_label']} {row['morphology_label']}")
    print(ascii_heatmap(image_from_row(row)))


In [ ]:
def transfer_report(name, train_items, feature_fn):
    _, predictions = nearest_centroid(train_items, feature_fn, target_test_rows)
    print(f"\n{name}")
    for item in predictions:
        print(item["sample_id"], item["true_label"], "->", item["predicted_label"], item["distances"])
    print("accuracy:", round(accuracy(predictions), 3))
    print("confusion matrix:", confusion(predictions))
    return predictions


source_raw_transfer_predictions = transfer_report(
    "Source raw-pixel prototypes -> target test",
    source_train_rows,
    lambda item: flatten(image_from_row(item)),
)
source_conv_transfer_predictions = transfer_report(
    "Source frozen conv features -> target test",
    source_train_rows,
    cnn_features,
)
adapt_raw_transfer_predictions = transfer_report(
    "One-shot target raw head -> target test",
    target_adapt_rows,
    lambda item: flatten(image_from_row(item)),
)
adapt_conv_transfer_predictions = transfer_report(
    "Frozen conv features + one-shot target head -> target test",
    target_adapt_rows,
    cnn_features,
)

transfer_metrics = {
    "source raw-pixel prototypes": round(accuracy(source_raw_transfer_predictions), 3),
    "source frozen conv features": round(accuracy(source_conv_transfer_predictions), 3),
    "one-shot target raw head": round(accuracy(adapt_raw_transfer_predictions), 3),
    "frozen conv features + one-shot target head": round(accuracy(adapt_conv_transfer_predictions), 3),
}
print("\nTransfer-learning style summary:")
for label, metric in transfer_metrics.items():
    print(label, "->", metric)

print("Interpretation:")
print("  Raw pixels drift when the target survey changes background level and exact coordinates.")
print("  The frozen convolution-style features still capture compact cores, horizontal disks, and double sources.")
print("  That is the notebook's smallest transfer-learning lesson: keep the reusable feature extractor, then adapt only a lightweight head.")


In [ ]:
WORKFLOW_DATA_PATH = Path("../../data/small/cnn_transfer_workflow_demo.csv").resolve()
WORKFLOW_SEED = 7
WORKFLOW_FILTERS = 3
WORKFLOW_EPOCHS = 1600
WORKFLOW_LEARNING_RATE = 0.015
WORKFLOW_TEMPERATURE = 4.0
WORKFLOW_THRESHOLD_MARGIN = 0.15

with WORKFLOW_DATA_PATH.open(newline="", encoding="utf-8") as handle:
    workflow_rows = []
    for raw in csv.DictReader(handle):
        parsed = {
            "sample_id": raw["sample_id"],
            "morphology_label": raw["morphology_label"],
            "domain_label": raw["domain_label"],
            "split_role": raw["split_role"],
            "quality_flag": raw["quality_flag"],
        }
        for row_index in range(IMAGE_SIZE):
            for col_index in range(IMAGE_SIZE):
                parsed[f"p{row_index}{col_index}"] = float(raw[f"p{row_index}{col_index}"])
        workflow_rows.append(parsed)


def workflow_image_from_row(row):
    return image_from_row(row)


workflow_source_train_rows = [row for row in workflow_rows if row["split_role"] == "source_train"]
workflow_target_adapt_rows = [row for row in workflow_rows if row["split_role"] == "target_adapt"]
workflow_target_validation_rows = [row for row in workflow_rows if row["split_role"] == "target_validation"]
workflow_target_test_rows = [row for row in workflow_rows if row["split_role"] == "target_test"]
workflow_review_rows = [row for row in workflow_rows if row["split_role"] == "review"]

print(f"Loaded {len(workflow_rows)} workflow cutouts from {WORKFLOW_DATA_PATH.name}")
print("split-role counts:", dict(Counter(row["split_role"] for row in workflow_rows)))
print("quality counts:", dict(Counter(row["quality_flag"] for row in workflow_rows)))
print("review ids:", [row["sample_id"] for row in workflow_review_rows])
print("Representative workflow cutouts:")
for sample_id in ["S_C0", "A_C0", "X_D0", "R_art"]:
    row = next(item for item in workflow_rows if item["sample_id"] == sample_id)
    print()
    print(f"{sample_id} {row['split_role']} {row['quality_flag']} {row['morphology_label']}")
    print(ascii_heatmap(workflow_image_from_row(row)))


In [ ]:
import random

workflow_label_order = sorted({row["morphology_label"] for row in workflow_rows})
workflow_label_to_index = {label: index for index, label in enumerate(workflow_label_order)}


def softmax(values):
    shifted = [value - max(values) for value in values]
    exp_values = [math.exp(value) for value in shifted]
    total = sum(exp_values)
    return [value / total for value in exp_values]


class TinyConv2dClassifier:
    def __init__(self, seed=WORKFLOW_SEED, num_filters=WORKFLOW_FILTERS, kernel_size=3):
        rng = random.Random(seed)
        self.kernel_size = kernel_size
        self.num_filters = num_filters
        self.output_size = IMAGE_SIZE - kernel_size + 1
        self.pool_size = 2
        self.pooled_side = self.output_size // self.pool_size
        self.feature_dim = num_filters * self.pooled_side * self.pooled_side
        self.filters = [
            [[rng.uniform(-0.25, 0.25) for _ in range(kernel_size)] for _ in range(kernel_size)]
            for _ in range(num_filters)
        ]
        self.filter_bias = [0.0 for _ in range(num_filters)]
        self.classifier = [
            [rng.uniform(-0.25, 0.25) for _ in range(self.feature_dim)]
            for _ in range(len(workflow_label_order))
        ]
        self.classifier_bias = [0.0 for _ in range(len(workflow_label_order))]

    def forward(self, image):
        conv_pre = []
        max_positions = []
        feature_vector = []

        for filter_index in range(self.num_filters):
            raw_map = []
            activated_map = []
            for row_index in range(self.output_size):
                raw_row = []
                activated_row = []
                for col_index in range(self.output_size):
                    value = self.filter_bias[filter_index]
                    for kernel_row in range(self.kernel_size):
                        for kernel_col in range(self.kernel_size):
                            value += (
                                self.filters[filter_index][kernel_row][kernel_col]
                                * image[row_index + kernel_row][col_index + kernel_col]
                            )
                    raw_row.append(value)
                    activated_row.append(max(0.0, value))
                raw_map.append(raw_row)
                activated_map.append(activated_row)
            conv_pre.append(raw_map)

            pooled_positions = []
            for pool_row in range(self.pooled_side):
                position_row = []
                for pool_col in range(self.pooled_side):
                    best_value = None
                    best_position = None
                    for delta_row in range(self.pool_size):
                        for delta_col in range(self.pool_size):
                            row_index = pool_row * self.pool_size + delta_row
                            col_index = pool_col * self.pool_size + delta_col
                            candidate = activated_map[row_index][col_index]
                            if best_value is None or candidate > best_value:
                                best_value = candidate
                                best_position = (row_index, col_index)
                    feature_vector.append(best_value)
                    position_row.append(best_position)
                pooled_positions.append(position_row)
            max_positions.append(pooled_positions)

        logits = []
        for class_index in range(len(workflow_label_order)):
            logits.append(
                self.classifier_bias[class_index]
                + sum(
                    self.classifier[class_index][feature_index] * feature_vector[feature_index]
                    for feature_index in range(self.feature_dim)
                )
            )
        probabilities = softmax(logits)
        predicted_index = max(range(len(probabilities)), key=lambda index: probabilities[index])
        return {
            "conv_pre": conv_pre,
            "max_positions": max_positions,
            "features": feature_vector,
            "probabilities": probabilities,
            "predicted_label": workflow_label_order[predicted_index],
            "confidence": probabilities[predicted_index],
        }

    def loss_and_gradients(self, row):
        image = workflow_image_from_row(row)
        summary = self.forward(image)
        target_index = workflow_label_to_index[row["morphology_label"]]
        probabilities = summary["probabilities"]
        loss = -math.log(max(probabilities[target_index], 1e-12))

        grad_filters = [
            [[0.0 for _ in range(self.kernel_size)] for _ in range(self.kernel_size)]
            for _ in range(self.num_filters)
        ]
        grad_filter_bias = [0.0 for _ in range(self.num_filters)]
        grad_classifier = [
            [0.0 for _ in range(self.feature_dim)]
            for _ in range(len(workflow_label_order))
        ]
        grad_classifier_bias = [0.0 for _ in range(len(workflow_label_order))]

        d_logits = probabilities[:]
        d_logits[target_index] -= 1.0
        for class_index in range(len(workflow_label_order)):
            grad_classifier_bias[class_index] += d_logits[class_index]
            for feature_index in range(self.feature_dim):
                grad_classifier[class_index][feature_index] += (
                    d_logits[class_index] * summary["features"][feature_index]
                )

        d_features = [
            sum(
                self.classifier[class_index][feature_index] * d_logits[class_index]
                for class_index in range(len(workflow_label_order))
            )
            for feature_index in range(self.feature_dim)
        ]

        feature_index = 0
        for filter_index in range(self.num_filters):
            d_relu = [[0.0 for _ in range(self.output_size)] for _ in range(self.output_size)]
            for pool_row in range(self.pooled_side):
                for pool_col in range(self.pooled_side):
                    row_index, col_index = summary["max_positions"][filter_index][pool_row][pool_col]
                    d_relu[row_index][col_index] += d_features[feature_index]
                    feature_index += 1

            for row_index in range(self.output_size):
                for col_index in range(self.output_size):
                    if summary["conv_pre"][filter_index][row_index][col_index] <= 0.0:
                        continue
                    gradient_value = d_relu[row_index][col_index]
                    if gradient_value == 0.0:
                        continue
                    grad_filter_bias[filter_index] += gradient_value
                    for kernel_row in range(self.kernel_size):
                        for kernel_col in range(self.kernel_size):
                            grad_filters[filter_index][kernel_row][kernel_col] += (
                                gradient_value * image[row_index + kernel_row][col_index + kernel_col]
                            )

        return loss, {
            "filters": grad_filters,
            "filter_bias": grad_filter_bias,
            "classifier": grad_classifier,
            "classifier_bias": grad_classifier_bias,
        }

    def train(self, items, epochs=WORKFLOW_EPOCHS, learning_rate=WORKFLOW_LEARNING_RATE):
        for _ in range(epochs):
            grad_filters = [
                [[0.0 for _ in range(self.kernel_size)] for _ in range(self.kernel_size)]
                for _ in range(self.num_filters)
            ]
            grad_filter_bias = [0.0 for _ in range(self.num_filters)]
            grad_classifier = [
                [0.0 for _ in range(self.feature_dim)]
                for _ in range(len(workflow_label_order))
            ]
            grad_classifier_bias = [0.0 for _ in range(len(workflow_label_order))]

            for row in items:
                _, gradients = self.loss_and_gradients(row)
                for filter_index in range(self.num_filters):
                    grad_filter_bias[filter_index] += gradients["filter_bias"][filter_index]
                    for kernel_row in range(self.kernel_size):
                        for kernel_col in range(self.kernel_size):
                            grad_filters[filter_index][kernel_row][kernel_col] += (
                                gradients["filters"][filter_index][kernel_row][kernel_col]
                            )
                for class_index in range(len(workflow_label_order)):
                    grad_classifier_bias[class_index] += gradients["classifier_bias"][class_index]
                    for feature_index in range(self.feature_dim):
                        grad_classifier[class_index][feature_index] += (
                            gradients["classifier"][class_index][feature_index]
                        )

            scale = 1.0 / len(items)
            for filter_index in range(self.num_filters):
                self.filter_bias[filter_index] -= learning_rate * grad_filter_bias[filter_index] * scale
                for kernel_row in range(self.kernel_size):
                    for kernel_col in range(self.kernel_size):
                        self.filters[filter_index][kernel_row][kernel_col] -= (
                            learning_rate * grad_filters[filter_index][kernel_row][kernel_col] * scale
                        )
            for class_index in range(len(workflow_label_order)):
                self.classifier_bias[class_index] -= learning_rate * grad_classifier_bias[class_index] * scale
                for feature_index in range(self.feature_dim):
                    self.classifier[class_index][feature_index] -= (
                        learning_rate * grad_classifier[class_index][feature_index] * scale
                    )


def workflow_feature_vector(row, model):
    return model.forward(workflow_image_from_row(row))["features"]


def workflow_distance_probabilities(feature_vector, prototypes, temperature=WORKFLOW_TEMPERATURE):
    distances = {
        label: euclidean_distance(feature_vector, prototypes[label])
        for label in workflow_label_order
    }
    probabilities = softmax([-distances[label] / temperature for label in workflow_label_order])
    return distances, {
        label: probability
        for label, probability in zip(workflow_label_order, probabilities)
    }


def workflow_route(row, confidence, ready_threshold):
    if row["quality_flag"] != "clean":
        return "manual_review"
    return "ready_for_science" if confidence >= ready_threshold else "manual_review"


workflow_model = TinyConv2dClassifier()
workflow_model.train(workflow_source_train_rows)

print("Learned tiny Conv2d filters on source_train:")
for filter_index, kernel in enumerate(workflow_model.filters):
    print(filter_index, [[round(value, 2) for value in row] for row in kernel])

source_train_predictions = [workflow_model.forward(workflow_image_from_row(row)) for row in workflow_source_train_rows]
source_train_accuracy = sum(
    prediction["predicted_label"] == row["morphology_label"]
    for row, prediction in zip(workflow_source_train_rows, source_train_predictions)
) / len(workflow_source_train_rows)
print("source-train accuracy:", round(source_train_accuracy, 3))

workflow_source_raw_centers = {}
for label in workflow_label_order:
    workflow_source_raw_centers[label] = centroid(
        [flatten(workflow_image_from_row(row)) for row in workflow_source_train_rows if row["morphology_label"] == label]
    )

print()
print("Source raw-pixel prototypes -> clean target test")
workflow_source_raw_predictions = []
for row in workflow_target_test_rows:
    vector = flatten(workflow_image_from_row(row))
    ranked = sorted(
        ((label, euclidean_distance(vector, workflow_source_raw_centers[label])) for label in workflow_label_order),
        key=lambda pair: pair[1],
    )
    workflow_source_raw_predictions.append({
        "sample_id": row["sample_id"],
        "true_label": row["morphology_label"],
        "predicted_label": ranked[0][0],
    })
    print(row["sample_id"], row["morphology_label"], "->", ranked[0][0], {label: round(distance, 3) for label, distance in ranked})
print("accuracy:", round(accuracy(workflow_source_raw_predictions), 3))

print()
print("Source-trained tiny Conv2d source head -> clean target test")
workflow_source_head_predictions = []
for row in workflow_target_test_rows:
    summary = workflow_model.forward(workflow_image_from_row(row))
    workflow_source_head_predictions.append({
        "sample_id": row["sample_id"],
        "true_label": row["morphology_label"],
        "predicted_label": summary["predicted_label"],
    })
    print(row["sample_id"], row["morphology_label"], "->", summary["predicted_label"], round(summary["confidence"], 4))
print("accuracy:", round(accuracy(workflow_source_head_predictions), 3))

workflow_target_prototypes = {
    label: centroid(
        [workflow_feature_vector(row, workflow_model) for row in workflow_target_adapt_rows if row["morphology_label"] == label]
    )
    for label in workflow_label_order
}

workflow_validation_predictions = []
for row in workflow_target_validation_rows:
    feature_vector = workflow_feature_vector(row, workflow_model)
    distances, probabilities = workflow_distance_probabilities(feature_vector, workflow_target_prototypes)
    predicted_label = max(probabilities, key=probabilities.get)
    workflow_validation_predictions.append({
        "sample_id": row["sample_id"],
        "true_label": row["morphology_label"],
        "predicted_label": predicted_label,
        "confidence": probabilities[predicted_label],
        "distances": distances,
    })

workflow_ready_threshold = max(
    0.0,
    min(item["confidence"] for item in workflow_validation_predictions) - WORKFLOW_THRESHOLD_MARGIN,
)
print()
print("Validation-calibrated routing threshold:")
for item in workflow_validation_predictions:
    print(
        item["sample_id"],
        item["true_label"],
        "->",
        item["predicted_label"],
        round(item["confidence"], 4),
        {label: round(distance, 3) for label, distance in item["distances"].items()},
    )
print("ready threshold =", round(workflow_ready_threshold, 4), "(weakest clean validation confidence - 0.15)")

print()
print("Frozen source backbone + target prototype head -> clean target test")
workflow_target_predictions = []
for row in workflow_target_test_rows:
    feature_vector = workflow_feature_vector(row, workflow_model)
    distances, probabilities = workflow_distance_probabilities(feature_vector, workflow_target_prototypes)
    predicted_label = max(probabilities, key=probabilities.get)
    confidence = probabilities[predicted_label]
    route = workflow_route(row, confidence, workflow_ready_threshold)
    workflow_target_predictions.append({
        "sample_id": row["sample_id"],
        "true_label": row["morphology_label"],
        "predicted_label": predicted_label,
        "route": route,
        "confidence": confidence,
    })
    print(
        row["sample_id"],
        row["morphology_label"],
        "->",
        predicted_label,
        round(confidence, 4),
        route,
        {label: round(distance, 3) for label, distance in distances.items()},
    )
print("accuracy:", round(accuracy(workflow_target_predictions), 3))
print("route counts:", dict(Counter(item["route"] for item in workflow_target_predictions)))

print()
print("Flagged review queue")
for row in workflow_review_rows:
    feature_vector = workflow_feature_vector(row, workflow_model)
    distances, probabilities = workflow_distance_probabilities(feature_vector, workflow_target_prototypes)
    predicted_label = max(probabilities, key=probabilities.get)
    confidence = probabilities[predicted_label]
    confidence_only_route = "ready_for_science" if confidence >= workflow_ready_threshold else "manual_review"
    print(
        row["sample_id"],
        row["quality_flag"],
        row["morphology_label"],
        "->",
        predicted_label,
        round(confidence, 4),
        "confidence_only=",
        confidence_only_route,
        "workflow=",
        workflow_route(row, confidence, workflow_ready_threshold),
        {label: round(distance, 3) for label, distance in distances.items()},
    )

print()
print("Workflow summary:")
print("  source raw-pixel prototypes accuracy =", round(accuracy(workflow_source_raw_predictions), 3))
print("  source-trained tiny Conv2d source-head accuracy =", round(accuracy(workflow_source_head_predictions), 3))
print("  adapted target prototype-head accuracy =", round(accuracy(workflow_target_predictions), 3))
print("  clean target auto-ready rows =", sum(item["route"] == "ready_for_science" for item in workflow_target_predictions), "/", len(workflow_target_predictions))
print("  flagged review rows captured =", len(workflow_review_rows), "/", len(workflow_review_rows))
print("Interpretation:")
print("  Training only on source leaves the compact-core target rows biased toward the wrong source-domain decision boundary.")
print("  Freezing the learned local backbone and rebuilding only a tiny target-domain prototype head recovers all clean target labels on this toy workflow.")
print("  The threshold is intentionally conservative, so the hardest clean double-source row X_D0 still routes to manual review.")
print("  Two flagged review rows would look ready on confidence alone, which is exactly why a separate quality gate still matters.")


In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"Optional matplotlib plot skipped: {exc}")
else:
    figure, axes = plt.subplots(2, 3, figsize=(8, 5))
    for axis, sample_id in zip(axes.flat, ["C01", "E01", "D01", "C03", "E03", "D03"]):
        row = next(item for item in rows if item["sample_id"] == sample_id)
        axis.imshow(image_from_row(row), cmap="magma", interpolation="nearest")
        axis.set_title(f"{sample_id}\n{row['morphology_label']}")
        axis.set_xticks([])
        axis.set_yticks([])
    figure.suptitle("Tiny galaxy-cutout teaching samples")
    figure.tight_layout()
    plt.show()


In [ ]:
try:
    import torch
except ModuleNotFoundError as exc:
    print(f"Optional PyTorch comparison skipped: {exc}")
else:
    torch.manual_seed(0)
    label_to_index = {label: index for index, label in enumerate(sorted({row["morphology_label"] for row in rows}))}
    x_train = torch.tensor([image_from_row(row) for row in train_rows], dtype=torch.float32).unsqueeze(1)
    y_train = torch.tensor([label_to_index[row["morphology_label"]] for row in train_rows], dtype=torch.long)
    x_test = torch.tensor([image_from_row(row) for row in test_rows], dtype=torch.float32).unsqueeze(1)
    y_test = torch.tensor([label_to_index[row["morphology_label"]] for row in test_rows], dtype=torch.long)

    model = torch.nn.Sequential(
        torch.nn.Conv2d(1, 4, kernel_size=3),
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(2),
        torch.nn.Flatten(),
        torch.nn.Linear(4, len(label_to_index)),
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
    loss_fn = torch.nn.CrossEntropyLoss()

    for _ in range(300):
        optimizer.zero_grad()
        logits = model(x_train)
        loss = loss_fn(logits, y_train)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        test_logits = model(x_test)
        predicted = test_logits.argmax(dim=1)
        test_accuracy = (predicted == y_test).float().mean().item()
        reverse_labels = {index: label for label, index in label_to_index.items()}
        print("Optional PyTorch tiny-CNN accuracy:", round(test_accuracy, 3))
        for row, pred_index in zip(test_rows, predicted.tolist()):
            print("torch", row["sample_id"], row["morphology_label"], "->", reverse_labels[pred_index])


In [ ]:
from part5_delivery_exports import export_ch31_cnn_delivery

export_ch31_cnn_delivery(globals())
